In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Read From .csv files").master("local[*]").getOrCreate()
spark

In [2]:
emp = spark.read.format("csv").load("csv_samples/emp.csv")
# 1 job will be created for the above code. The task will read the metadata of the file.
# spark identify the meta data but not the headers

In [3]:
emp.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)



In [4]:
# this will tell the spark that the first row of the csv file is the header and it will use that as the column names
emp = spark.read.format("csv").option("header", True).load("csv_samples/emp.csv")
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [5]:
# inferSchema will tell the spark to infer the data types of the columns based on the data in the csv file. 
# This will create a job to read the data and infer the schema.
emp = spark.read.format("csv").option("header", True).option("inferSchema", True).load("csv_samples/emp.csv")
emp.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: timestamp (nullable = true)



In [6]:
# this will create a job to read 20 rows of data from the csv because 1st one is the header
emp.show() 

+-----------+-------------+-------------+---+------+------+-------------------+
|employee_id|department_id|         name|age|gender|salary|          hire_date|
+-----------+-------------+-------------+---+------+------+-------------------+
|          1|          101|     John Doe| 30|  Male| 50000|2015-01-01 00:00:00|
|          2|          101|   Jane Smith| 25|Female| 45000|2016-02-15 00:00:00|
|          3|          102|    Bob Brown| 35|  Male| 55000|2014-05-01 00:00:00|
|          4|          102|    Alice Lee| 28|Female| 48000|2017-09-30 00:00:00|
|          5|          103|    Jack Chan| 40|  Male| 60000|2013-04-01 00:00:00|
|          6|          103|    Jill Wong| 32|Female| 52000|2018-07-01 00:00:00|
|          7|          101|James Johnson| 42|  Male| 70000|2012-03-15 00:00:00|
|          8|          102|     Kate Kim| 29|Female| 51000|2019-10-01 00:00:00|
|          9|          103|      Tom Tan| 33|  Male| 58000|2016-06-01 00:00:00|
|         10|          104|     Lisa Lee

In [7]:
# forcing custom schema so this will not create a job to read the data and infer the schema. 
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"
emp = spark.read.format("csv").schema(_schema).load("csv_samples/emp.csv")
# because spark dosen't need to go into the file and look for the meta data
emp.show()

#1st row is null because we are forcing the schema and the first row is the header. 
# So spark will treat the header as data and will try to convert it to the data type specified in the schema. 
# Since the header is not a valid int, it will be converted to null.

+-----------+-------------+-------------+----+------+-------+----------+
|employee_id|department_id|         name| age|gender| salary| hire_date|
+-----------+-------------+-------------+----+------+-------+----------+
|       null|         null|         name|null|gender|   null|      null|
|          1|          101|     John Doe|  30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith|  25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown|  35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee|  28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan|  40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong|  32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson|  42|  Male|70000.0|2012-03-15|
|          8|          102|     Kate Kim|  29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan|  33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee|  27|Femal

In [8]:
# PERMISSIVE (default) — bad field becomes null
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"

emp_new = (spark.read.format("csv")
           .option("header", True)
           .schema(_schema)
           .load("csv_samples/emp_new.csv"))
emp_new.show()

+-----------+-------------+-------------+---+------+-------+----------+
|employee_id|department_id|         name|age|gender| salary| hire_date|
+-----------+-------------+-------------+---+------+-------+----------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|
|          7|          101|James Johnson| 42|  Male|   null|2012-03-15|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|      Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|     Lisa Lee| 27|Female|47000.0|      null|
|         11|          104|   David Park| 38|  Male|65000.0|    

In [9]:
# using PERMISSIVE mode, we need to add a new column to the schema to capture the bad data.
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date, _corrupt_record string"
emp_new_p = (spark.read.format("csv")
           .option("header", True)
           .option("mode", "PERMISSIVE")   # this is the default, shown here to be explicit
           .schema(_schema)
           .load("csv_samples/emp_new.csv"))
emp_new_p.show()

+-----------+-------------+-------------+---+------+-------+----------+--------------------+
|employee_id|department_id|         name|age|gender| salary| hire_date|     _corrupt_record|
+-----------+-------------+-------------+---+------+-------+----------+--------------------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|                null|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|                null|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|                null|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|                null|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|                null|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|                null|
|          7|          101|James Johnson| 42|  Male|   null|2012-03-15|007,101,James Joh...|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01

In [10]:
# to select the badrecords only
emp_new_p.where("_corrupt_record is not null").show()

+-----------+-------------+-------------+---+------+-------+----------+--------------------+
|employee_id|department_id|         name|age|gender| salary| hire_date|     _corrupt_record|
+-----------+-------------+-------------+---+------+-------+----------+--------------------+
|          7|          101|James Johnson| 42|  Male|   null|2012-03-15|007,101,James Joh...|
|         11|          104|   David Park| 38|  Male|65000.0|      null|011,104,David Par...|
+-----------+-------------+-------------+---+------+-------+----------+--------------------+



In [11]:
# how to set custom name for the _corrupt_record column
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date, _bad_data string"
emp_new_p = (spark.read.format("csv")
           .option("header", True)
           .option("columnNameOfCorruptRecord", "_bad_data")
           .option("mode", "PERMISSIVE")
           .schema(_schema)
           .load("csv_samples/emp_new.csv"))
emp_new_p.show()

+-----------+-------------+-------------+---+------+-------+----------+--------------------+
|employee_id|department_id|         name|age|gender| salary| hire_date|           _bad_data|
+-----------+-------------+-------------+---+------+-------+----------+--------------------+
|          1|          101|     John Doe| 30|  Male|50000.0|2015-01-01|                null|
|          2|          101|   Jane Smith| 25|Female|45000.0|2016-02-15|                null|
|          3|          102|    Bob Brown| 35|  Male|55000.0|2014-05-01|                null|
|          4|          102|    Alice Lee| 28|Female|48000.0|2017-09-30|                null|
|          5|          103|    Jack Chan| 40|  Male|60000.0|2013-04-01|                null|
|          6|          103|    Jill Wong| 32|Female|52000.0|2018-07-01|                null|
|          7|          101|James Johnson| 42|  Male|   null|2012-03-15|007,101,James Joh...|
|          8|          102|     Kate Kim| 29|Female|51000.0|2019-10-01

In [15]:
# DROPMALFORMED — the whole bad row disappears
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"
emp_drop = (spark.read.format("csv")
            .option("header", True)
            .option("mode", "DROPMALFORMED")
            .schema(_schema)
            .load("csv_samples/emp_new.csv"))
emp_drop.show()  
# employee_id 7 is gone,11 is gone, 
# 10 is there because empty hire_date is not considered malformed, only the bad data in the row is considered malformed.

+-----------+-------------+-----------+---+------+-------+----------+
|employee_id|department_id|       name|age|gender| salary| hire_date|
+-----------+-------------+-----------+---+------+-------+----------+
|          1|          101|   John Doe| 30|  Male|50000.0|2015-01-01|
|          2|          101| Jane Smith| 25|Female|45000.0|2016-02-15|
|          3|          102|  Bob Brown| 35|  Male|55000.0|2014-05-01|
|          4|          102|  Alice Lee| 28|Female|48000.0|2017-09-30|
|          5|          103|  Jack Chan| 40|  Male|60000.0|2013-04-01|
|          6|          103|  Jill Wong| 32|Female|52000.0|2018-07-01|
|          8|          102|   Kate Kim| 29|Female|51000.0|2019-10-01|
|          9|          103|    Tom Tan| 33|  Male|58000.0|2016-06-01|
|         10|          104|   Lisa Lee| 27|Female|47000.0|      null|
|         12|          105| Susan Chen| 31|Female|54000.0|2017-02-15|
|         13|          106|  Brian Kim| 45|  Male|75000.0|2011-07-01|
|         14|       

## `count()` does not see malformed rows

Spark prunes columns it does not need. `count()` needs **no** column value, so the CSV parser never converts a single field — which means it never hits the `NumberFormatException`, never flags the record, and never drops it.

So on the *same* DataFrame:

| Action | Result | Why |
|---|---|---|
| `emp_drop.count()` | **20** | no column parsed → nothing looks malformed |
| `len(emp_drop.collect())` | **19** | every column parsed → bad row dropped |

**Lesson:** never validate a row count with `count()` on a raw file read. Cache/materialise it first, or count something that forces parsing.


In [16]:
# the same DataFrame gives two different answers
print("count()  :", emp_drop.count())          # 20 — no column is parsed, so nothing looks malformed
print("collect():", len(emp_drop.collect()))   # 18 — the bad row really is dropped

count()  : 20
collect(): 18


In [ ]:
# FAILFAST — refuse to read anything if a record is malformed
_schema = "employee_id int, department_id int, name string, age int, gender string, salary double, hire_date date"
emp_fail = (spark.read.format("csv")
            .option("header", True)
            .option("mode", "FAILFAST")
            .schema(_schema)
            .load("csv_samples/emp_new.csv"))

# note: .load() does NOT fail — reading is lazy. It blows up on the ACTION below.On show() the job is created 
# at the time of job execution, the job fails
try:
    emp_fail.show()
except Exception as e:
    print(str(e)[:400])

# SparkException: Malformed records are detected in record parsing. Parse Mode: FAILFAST.
#   Caused by: BadRecordException: java.lang.NumberFormatException: For input string: "Low"

An error occurred while calling o142.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 18.0 failed 1 times, most recent failure: Lost task 0.0 in stage 18.0 (TID 17) (f281bc21d442 executor driver): org.apache.spark.SparkException: Malformed records are detected in record parsing. Parse Mode: FAILFAST. To process malformed records as null result, try s


In [23]:
# Databricks: bad rows go to /tmp/bad_records as JSON and are removed from the DataFrame.
# This open-source Spark 3.3: option is ignored → behaves like PERMISSIVE, no folder is created.
emp_bad = (spark.read.format("csv")
           .option("header", True)
           .option("badRecordsPath", "/tmp/bad_records")
           .schema(_schema)
           .load("csv_samples/emp_new.csv"))
emp_bad.filter("employee_id = 7").show()   # still here, salary null

+-----------+-------------+-------------+---+------+------+----------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|     _corrupt_record|
+-----------+-------------+-------------+---+------+------+----------+--------------------+
|          7|          101|James Johnson| 42|  Male|  null|2012-03-15|007,101,James Joh...|
+-----------+-------------+-------------+---+------+------+----------+--------------------+



In [19]:
# BONUS TIP
# Multiple options can be passed as a dictionary to the .options() method.

_options = {
    "header" : "true",
    "inferSchema" : "true",
    "mode" : "PERMISSIVE"
}
# *variuable names can be used for the dictionary, but the keys must match the option names exactly.
df = (spark.read.format("csv").options(**_options).load("csv_samples/emp.csv"))
df.show()

+-----------+-------------+-------------+---+------+------+-------------------+
|employee_id|department_id|         name|age|gender|salary|          hire_date|
+-----------+-------------+-------------+---+------+------+-------------------+
|          1|          101|     John Doe| 30|  Male| 50000|2015-01-01 00:00:00|
|          2|          101|   Jane Smith| 25|Female| 45000|2016-02-15 00:00:00|
|          3|          102|    Bob Brown| 35|  Male| 55000|2014-05-01 00:00:00|
|          4|          102|    Alice Lee| 28|Female| 48000|2017-09-30 00:00:00|
|          5|          103|    Jack Chan| 40|  Male| 60000|2013-04-01 00:00:00|
|          6|          103|    Jill Wong| 32|Female| 52000|2018-07-01 00:00:00|
|          7|          101|James Johnson| 42|  Male| 70000|2012-03-15 00:00:00|
|          8|          102|     Kate Kim| 29|Female| 51000|2019-10-01 00:00:00|
|          9|          103|      Tom Tan| 33|  Male| 58000|2016-06-01 00:00:00|
|         10|          104|     Lisa Lee

In [20]:
spark.stop()